# Golf Course Feature Mapper — Colab Quickstart

End-to-end: OSM course discovery → Esri imagery (training + target) → YOLO training → SAM 2.1 → OSM export.

**Before running:** Set runtime to GPU — **Runtime → Change runtime type → T4 GPU**

**Licensing:** Esri World Imagery tiles may be traced for OSM but must not be redistributed.

In [ ]:
# ── Cell 1: GPU check ──────────────────────────────────────────────────────
import subprocess, sys
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print('GPU detected:'); print(r.stdout[:500])
else:
    print('WARNING: No GPU — Runtime → Change runtime type → GPU')
try:
    import torch
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'CUDA: {p.name}  {p.total_memory/1e9:.1f} GB')
    else:
        print('CUDA not available — install GPU runtime')
except ImportError:
    print('PyTorch not yet installed')

In [ ]:
# ── Cell 2: Clone repo + install dependencies ──────────────────────────────
import os, pathlib
REPO = '/content/golf_course_feature_mapping'
if not pathlib.Path(REPO).exists():
    # Replace with your actual repo URL:
    subprocess.run(['git', 'clone', 'https://github.com/YOUR_USER/golf_course_feature_mapping.git', REPO], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# ── Cell 3: Write versions.lock ────────────────────────────────────────────
import pathlib
result = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
pathlib.Path('versions.lock').write_text(result.stdout)
print(f'versions.lock: {len(result.stdout.splitlines())} packages')

In [ ]:
# ── Cell 4: (Optional) Mount Google Drive ──────────────────────────────────
USE_DRIVE = False   # Set True to persist data across sessions
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Update data.* paths in config.yaml to /content/drive/MyDrive/golf_mapper/...')
else:
    print('Using ephemeral /content/golf_mapper — data will not survive session restart')

In [ ]:
# ── Cell 5: Load configuration ─────────────────────────────────────────────
import logging
from golf_mapper import load_config, setup_logging, set_seeds

setup_logging(level=logging.INFO)
cfg = load_config('config.yaml')
set_seeds(cfg.training.seed)
cfg.data.make_dirs()

print(f'AOI bbox     : {cfg.aoi.bbox}')
print(f'Target course: {cfg.aoi.osm_id}')
print(f'Model        : {cfg.model.variant}')
print(f'Zoom         : {cfg.imagery.zoom}')
print(f'Tile size    : {cfg.labels.tile_size}px')
print(f'Min features : {cfg.labels.min_features}')
print(f'Weighting    : {cfg.training.weighting_mode} (σ={cfg.training.sigma_km} km)')
print(f'Config hash  : {cfg.config_hash()}')
print(f'Classes      : {cfg.class_names}')

In [ ]:
# ── Cell 6: Stage 1 — Course discovery & eligibility filter ───────────────
# Queries Overpass for all leisure=golf_course elements in the bbox,
# counts their tagged features, and marks courses with >= min_features
# as qualifying training courses.
from golf_mapper.discovery import discover_courses

manifest_gdf = discover_courses(cfg)

n_train = manifest_gdf['qualifies_as_training'].sum()
print(f'\nDiscovered {len(manifest_gdf)} course(s), {n_train} qualify for training:')
print(manifest_gdf[['course_id', 'name', 'qualifies_as_training', 'total_features']].to_string())

In [ ]:
# ── Cell 7: Stage 2 — Fetch target course boundary + its imagery ───────────
# The target course (cfg.aoi.osm_id = Augusta National) is the course we want
# to MAP. Its imagery is downloaded here for use at inference time (Cell 10).
#
# Training course imagery is downloaded automatically inside build_dataset
# (Cell 8) via fetch_all_training_imagery — one GeoTIFF per training course.
from golf_mapper.discovery import _extract_boundary
from golf_mapper.imagery import fetch_course_imagery
from golf_mapper.osm import OverpassClient, build_boundary_query, parse_osm_ref

client = OverpassClient(cfg.overpass, cache_dir=cfg.data.overpass_cache)
osm_type, osm_id = parse_osm_ref(cfg.aoi.osm_id)
data = client.query(
    build_boundary_query(osm_type, osm_id),
    cache_key=f'boundary:{cfg.aoi.osm_id}',
)
target_boundary, _ = _extract_boundary(data, osm_type, osm_id, cfg.aoi.name)

target_tif, meta = fetch_course_imagery(target_boundary, cfg.aoi.osm_id, cfg)
print(f'Target GeoTIFF : {target_tif.name}')
print(f'Size           : {meta["file_size_mb"]:.1f} MB')
print(f'Dims           : {meta["width_px"]} x {meta["height_px"]} px')
print(f'GSD            : {meta["ground_resolution_m"]:.3f} m/px')
print(f'CRS            : {meta["crs"]}')

In [ ]:
# ── Cell 8: Stage 3 — Download training imagery + build tiled dataset ──────
# build_dataset automatically calls fetch_all_training_imagery(), which
# downloads a separate Esri GeoTIFF for EACH qualifying training course
# found in the manifest. OSM feature geometries are fetched from Overpass
# and rasterized onto each course's own imagery to produce pixel-aligned
# YOLO-seg labels. Everything is tiled to TILE_SIZE x TILE_SIZE patches
# and split by course (no tile from course X in both train and val).
#
# Expect this cell to take a while: one Overpass + one imagery download
# per training course (~50-150 courses in the default bbox).
from golf_mapper.labels import build_dataset

training_manifest = manifest_gdf[manifest_gdf['qualifies_as_training']]
if training_manifest.empty:
    raise RuntimeError(
        'No qualifying training courses found. '
        'Check the bbox in config.yaml or lower min_features.'
    )

# No imagery_paths or boundary_geoms needed — build_dataset fetches everything.
dataset_dir = build_dataset(training_manifest, cfg)

import yaml
with open(dataset_dir / 'data.yaml') as f:
    data_meta = yaml.safe_load(f)
print(f'\nDataset root : {dataset_dir}')
print(f'Classes      : {data_meta["names"]}')
print(f'nc           : {data_meta["nc"]}')
import pathlib
for split in ('train', 'val', 'test'):
    n = len(list((dataset_dir / 'images' / split).glob('*.png')))
    print(f'  {split:5s} tiles: {n}')

In [ ]:
# ── Cell 9: Stage 4 — Training ────────────────────────────────────────────
# In regional mode: short global pre-train on all courses, then fine-tune
# with tiles from courses near Augusta weighted more heavily.
from golf_mapper.train import train_model

target_centroid = (target_boundary.centroid.x, target_boundary.centroid.y)

best_checkpoint = train_model(
    cfg,
    dataset_dir=cfg.data.dataset_dir,
    target_centroid=target_centroid,
    manifest_gdf=manifest_gdf,
)
print(f'Best checkpoint: {best_checkpoint}')

In [ ]:
# ── Cell 10: Stage 5 — Inference + SAM 2.1 ───────────────────────────────
# Runs the trained model over the TARGET course (Augusta), not training courses.
from golf_mapper.infer import run_inference

pred_gdf = run_inference(
    tif_path=target_tif,
    boundary=target_boundary,
    model_path=best_checkpoint,
    cfg=cfg,
)
print(f'Predictions: {len(pred_gdf)} polygons')
print(pred_gdf.groupby('class_name').size())

In [ ]:
# ── Cell 11: Stage 6 — Confirmation gate + export ─────────────────────────
# Renders an interactive Folium map and REQUIRES explicit 'y' to write outputs.
from golf_mapper.export import export_course

outputs = export_course(
    raw_gdf=pred_gdf,
    course_id=cfg.aoi.osm_id,
    boundary=target_boundary,
    cfg=cfg,
    model_path=best_checkpoint,
    course_name=cfg.aoi.name,
    interactive=True,
)
if outputs:
    print(f"GeoJSON     : {outputs['geojson']}")
    print(f"OSM XML     : {outputs['osm']}")
    print(f"Provenance  : {outputs['provenance']}")
else:
    print('Export skipped (no confirmation or no valid predictions).')

In [ ]:
# ── Cell 12: Stage 7 — Evaluation / QA ───────────────────────────────────
# Compare predictions against the existing OSM features on Augusta as a
# sanity check (lower bound — the model trained on Augusta's own labels).
# For a proper held-out eval, point this at a course NOT in the training set.
from golf_mapper.evaluate import evaluate_course, write_evaluation_report
from golf_mapper.labels import fetch_course_features
from golf_mapper.osm import osm_element_to_area_id

osm_type, osm_id = parse_osm_ref(cfg.aoi.osm_id)
area_id = osm_element_to_area_id(osm_type, osm_id)
ref_gdf = fetch_course_features(client, area_id, cfg.aoi.osm_id, cfg)

metrics = evaluate_course(pred_gdf, ref_gdf, cfg, iou_threshold=0.5)
csv_path, md_path = write_evaluation_report(
    metrics, cfg.data.output_dir, course_id=cfg.aoi.osm_id
)
print(open(md_path).read())

## Uploading to OSM

1. Open `output/<course>.osm` in JOSM (`File → Open`).
2. Download existing OSM data (`File → Update Data`).
3. Review every polygon — fix errors, add hole numbers, names, etc.
4. Upload with your OSM account. **Never upload without human review.**

Suggested changeset comment: *Traced from Esri World Imagery with golf-mapper (ML-assisted); imagery © Esri*